In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.41.2
!pip uninstall -y peft

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection

!pip install transformers datasets scikit-learn -q

import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/balanced_dataset_12k.csv')

In [ ]:
df.head()

,headline,label,confidence
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,0.65
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,0.85
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,0.75
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,0.45
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,0.45


In [ ]:
df = df.drop(columns=['confidence'], axis = 1)

In [ ]:
import re

def clean_text(text):
    text = str(text)

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df['text'] = df['headline'].apply(lambda x: clean_text(x))

In [ ]:
df.head()

,headline,label,text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts"
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet"
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress"
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline"
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP


In [ ]:
from huggingface_hub import login
login(token="######################")

In [ ]:
texts = df["headline"].tolist()
labels = df["label"].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, stratify=labels, random_state=42
)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset   = Dataset.from_dict({"text": val_texts, "label": val_labels})

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128   # 🔥 optimized for speed
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10605 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

In [ ]:
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=2,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using:", device)

Using: cuda


In [ ]:
for param in model.distilbert.transformer.layer[:6].parameters():
    param.requires_grad = False

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_major",

    # 🔥 GPU optimization
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    fp16=False,

    # training control
    learning_rate=2e-5,
    num_train_epochs=50,
    warmup_ratio=0.1,
    weight_decay=0.01,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
def encode_labels(example):
    label_map = {
        "non-sarcastic": 0,
        "sarcastic": 1
    }
    example["label"] = label_map[example["label"]]
    return example

In [ ]:
train_dataset = train_dataset.map(encode_labels)
val_dataset = val_dataset.map(encode_labels)

Map:   0%|          | 0/10605 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset[0])

{'label': tensor(0), 'input_ids': tensor([  101, 24954,  2097,  2272,  2000,  2373,  2065,  5152,  4494,  2024,
         4055,  1024,  9815, 24281,  2072,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,  

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        # Overall
        "accuracy": accuracy_score(labels, preds),

        # Binary class metrics (positive = sarcastic)
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),

        # Macro (treat classes equally)
        "precision_macro": precision_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        "f1_macro": f1_score(labels, preds, average="macro"),

        # Weighted (accounts for imbalance)
        "precision_weighted": precision_score(labels, preds, average="weighted"),
        "recall_weighted": recall_score(labels, preds, average="weighted"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# STEP 1 — FIX LABELS FIRST
# def encode_labels(example):
#     label_map = {
#         "non-sarcastic": 0,
#         "sarcastic": 1
#     }
#     example["label"] = label_map[example["label"]]
#     return example

# train_dataset = train_dataset.map(encode_labels)
# val_dataset = val_dataset.map(encode_labels)

# STEP 2 — REMOVE EXTRA COLUMNS (IMPORTANT)
train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ["input_ids", "attention_mask", "label"]]
)

val_dataset = val_dataset.remove_columns(
    [col for col in val_dataset.column_names if col not in ["input_ids", "attention_mask", "label"]]
)

# STEP 3 — SET FORMAT
train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")

# STEP 4 — NOW CREATE TRAINER (AFTER FIXES)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# STEP 5 — TRAIN
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,No log,0.690938,0.513147,0.564103,0.112054,0.186969,0.535818,0.512807,0.419756,0.535794,0.513147,0.419953
2,0.692300,0.682189,0.611535,0.716172,0.368421,0.486547,0.645757,0.611329,0.587066,0.645697,0.611535,0.587151
3,0.684200,0.666319,0.661578,0.689243,0.587436,0.634280,0.665153,0.661515,0.659682,0.665133,0.661578,0.659703
4,0.667700,0.640878,0.673452,0.713389,0.578947,0.639175,0.679804,0.673372,0.670478,0.679776,0.673452,0.670505
5,0.637600,0.609655,0.676845,0.681818,0.662139,0.671835,0.676988,0.676832,0.676769,0.676984,0.676845,0.676774
6,0.595400,0.587504,0.685327,0.689236,0.674024,0.681545,0.685414,0.685317,0.685282,0.685411,0.685327,0.685285
7,0.595400,0.576338,0.692112,0.690878,0.694397,0.692633,0.692117,0.692114,0.692111,0.692118,0.692112,0.692111
8,0.565800,0.571710,0.694656,0.697074,0.687606,0.692308,0.694691,0.694651,0.694639,0.694689,0.694656,0.694641
9,0.544000,0.569760,0.699746,0.701544,0.694397,0.697952,0.699765,0.699741,0.699735,0.699764,0.699746,0.699736
10,0.527600,0.568362,0.699746,0.702936,0.691002,0.696918,0.699801,0.699738,0.699719,0.699799,0.699746,0.699722


TrainOutput(global_step=2100, training_loss=0.37739468029567175, metrics={'train_runtime': 5338.3453, 'train_samples_per_second': 99.329, 'train_steps_per_second': 0.393, 'total_flos': 1.7560209534336e+16, 'train_loss': 0.37739468029567175, 'epoch': 50.0})

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.5767192244529724, 'eval_accuracy': 0.7039864291772688, 'eval_precision': 0.6993355481727574, 'eval_recall': 0.7147707979626485, 'eval_f1': 0.7069689336691856, 'eval_precision_macro': 0.7040871848316127, 'eval_recall_macro': 0.7039955684728496, 'eval_f1_macro': 0.7039557607506168, 'eval_precision_weighted': 0.7040912150577864, 'eval_recall_weighted': 0.7039864291772688, 'eval_f1_weighted': 0.7039532050483958, 'eval_runtime': 4.8412, 'eval_samples_per_second': 243.537, 'eval_steps_per_second': 1.033, 'epoch': 50.0}


In [ ]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/models/sarcasm_bert_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/models/sarcasm_bert_model")

('/content/drive/MyDrive/Colab Notebooks/models/sarcasm_bert_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/models/sarcasm_bert_model/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/models/sarcasm_bert_model/vocab.txt',
 '/content/drive/MyDrive/Colab Notebooks/models/sarcasm_bert_model/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/models/sarcasm_bert_model/tokenizer.json')

In [ ]:
from transformers import pipeline

clf = pipeline("text-classification", model="sarcasm_bert_model")

clf("Government promises transparency, deletes all records")

OSError: sarcasm_bert_model is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
from sklearn.metrics import classification_report

preds_output = trainer.predict(val_dataset)
preds = np.argmax(preds_output.predictions, axis=1)

print(classification_report(val_labels, preds, target_names=["non-sarcastic", "sarcastic"]))